In [ ]:
# Notebook: Fourier Reconstruction of a Square Pulse Sequence
# Author: Adapted for Springer-style presentation
# Fourier expansion:
# x(t) = a_0 + sum_{n=1}^{N} [ 2 * a_n * cos(n * omega * t) ]
# where a_0 = A * tau / T and a_n = (A * tau / T) * (sin(n * omega * tau / 2) / (n * omega * tau / 2))

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import BoundedIntText, FloatSlider, HBox, VBox, Output
from IPython.display import display


# ==========================
# Parameters
# ==========================

t = np.linspace(-5, 5, 2000)


# ==========================
# Square pulse signal
# ==========================

def x_square(t, A=1, tau=2, T=6):
    # Περιοδικό τετραγωνικό σήμα πλάτους Α, διάρκειας παλμού tau και περιόδου Τ
    # Τοποθετούμε το παλμό στο διάστημα [-tau/2, tau/2] για κάθε περίοδο
    t_mod = (t + T/2) % T - T/2
    return np.where(np.abs(t_mod) <= tau/2, A, 0.0)


# ==========================
# Fourier expansion
# ==========================

def x_fourier_square(t, N, A=1, tau=2, T=6):
    omega = 2 * np.pi / T
    
    # a_0 συντελεστής (μέση τιμή)
    a0 = (A * tau) / T
    xf = np.full_like(t, a0)
    
    # Προσθήκη των αρμονικών (επειδή είναι άρτιο σήμα, έχουμε μόνο όρους συνημιτόνου)
    for n in range(1, N + 1):
        # Συντελεστής a_n
        arg = n * omega * tau / 2
        # Αποφυγή διαίρεσης με το 0 (αν και για n>=1 δεν μηδενίζεται αν tau/T είναι πεπερασμένο)
        sinc_val = np.sin(arg) / arg if arg != 0 else 1.0
        an = (A * tau / T) * sinc_val
        
        xf += 2 * an * np.cos(n * omega * t)
        
    return xf


# ==========================
# Controls
# ==========================

N = BoundedIntText(
    value=5,
    min=1,
    max=50,
    description="N"
)

A = FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description="A"
)

tau = FloatSlider(
    value=2.0,
    min=0.5,
    max=4.0,
    step=0.1,
    description="tau"
)

T_slider = FloatSlider(
    value=6.0,
    min=2.0,
    max=10.0,
    step=0.1,
    description="T"
)

out = Output()


# ==========================
# Plot
# ==========================

def plot(*args):

    with out:

        out.clear_output(wait=True)

        # Διασφάλιση ότι η περίοδος Τ είναι πάντα μεγαλύτερη από το tau
        if tau.value >= T_slider.value:
            print("Σφάλμα: Η περίοδος T πρέπει να είναι μεγαλύτερη από τη διάρκεια tau.")
            return

        xx = x_square(t, A.value, tau.value, T_slider.value)
        xf = x_fourier_square(t, N.value, A.value, tau.value, T_slider.value)

        fig, ax = plt.subplots(figsize=(9, 4))

        ax.plot(
            t, xx,
            'r',
            linewidth=3,
            label="Original square pulse"
        )

        ax.plot(
            t, xf,
            'b',
            linewidth=2,
            label=f"Fourier expansion (N={N.value})"
        )

        ax.set_xlabel(r"$t$")
        ax.set_ylabel("Amplitude")
        ax.grid(True)
        ax.legend()

        ax.set_title(
            "Square Pulse Signal Fourier Reconstruction"
        )

        display(fig)
        plt.close(fig)


N.observe(plot, names="value")
A.observe(plot, names="value")
tau.observe(plot, names="value")
T_slider.observe(plot, names="value")


display(
    VBox(
        [
            HBox([N, A]),
            HBox([tau, T_slider]),
            out
        ]
    )
)

plot()